In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import brainmass
import brainstate
import brainunit as u
import jax
import jax.numpy as jnp
import numpy as np
brainstate.environ.set(dt=0.1 * u.ms)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


# Adding Coupling

This tutorial focuses on coupling mechanisms for connecting brain regions in networks.

See {doc}`/tutorials/04_building_a_network` for general network setup. This guide covers coupling details.

:::{note}
Every code cell below is executed as part of the documentation build, so the examples
show the real construction API and produce real output. In `brainmass`, a coupling is
always constructed from *prefetch* references to a node module's state and evaluated with
`coupling.update()` inside a `brainstate.environ.context`, as in the first example and in
{doc}`/getting_started/quickstart`. The shorthand `C_i = k * ...` formulas in each section describe the
mathematics; the code shows the corresponding construction.
:::

## Coupling Types

### Diffusive Coupling

Drives nodes toward their neighbors' states:

$$
C_i = k \sum_j W_{ij} (x_j - x_i)
$$

A `DiffusiveCoupling` is built from two prefetch reads of the node state: a (possibly
delayed) read of every *source* node and a read of each *target* (self) node. It returns a
per-node coupling current:

In [2]:
import brainmass
import braintools
import brainstate
import brainunit as u
import jax.numpy as jnp
import numpy as np

brainstate.environ.set(dt=0.1 * u.ms)
N = 10

nodes = brainmass.HopfStep(in_size=N, a=0.25, w=0.2)

W = jnp.ones((N, N)) * 0.1  # connectivity matrix
W = W.at[jnp.diag_indices(N)].set(0.)  # no self-connections

# Each target reads every source's (delayed) ``x`` -> shape (N, N)
delays = jnp.ones((N, N)) * (1.0 * u.ms)
src_idx = np.tile(np.arange(N)[None, :], (N, 1))
x_src = nodes.prefetch_delay('x', delays, src_idx, init=braintools.init.Constant(0.0))
x_self = nodes.prefetch('x')

coupling = brainmass.DiffusiveCoupling(x_src, x_self, conn=W, k=0.5)

brainstate.nn.init_all_states(nodes)
brainstate.nn.init_all_states(coupling)

with brainstate.environ.context(i=0, t=0. * u.ms):
    coupled_input = coupling.update()  # (N,) per-node coupling current

### Additive Coupling

Weighted sum of neighbor inputs:

$$
C_i = k \sum_j W_{ij} x_j
$$

In [3]:
# Additive coupling reads a single (delayed) source state and weights it by ``conn``.
# It reuses the ``nodes`` / ``W`` / ``delays`` / ``src_idx`` defined above.
x_src_add = nodes.prefetch_delay('x', delays, src_idx, init=braintools.init.Constant(0.0))
coupling = brainmass.AdditiveCoupling(x_src_add, conn=W, k=0.3)

brainstate.nn.init_all_states(coupling)
with brainstate.environ.context(i=0, t=0. * u.ms):
    coupled_input = coupling.update()  # (N,) per-node coupling current
coupled_input.shape

(10,)

### When to Use Each

- **Diffusive**: Most common for brain networks; models synchronization
- **Additive**: Direct input summation; simpler but less realistic

## Coupling Strength

The global coupling parameter `k` controls network integration:

In [4]:
# Each ``DiffusiveCoupling`` needs its own pair of prefetch reads. The only
# difference between these three is the global coupling strength ``k``.
def make_diffusive(k):
    x_src = nodes.prefetch_delay('x', delays, src_idx, init=braintools.init.Constant(0.0))
    x_self = nodes.prefetch('x')
    return brainmass.DiffusiveCoupling(x_src, x_self, conn=W, k=k)

coupling_weak = make_diffusive(0.01)    # weak coupling: nearly independent regions
coupling_mod = make_diffusive(0.2)      # moderate coupling: balanced
coupling_strong = make_diffusive(1.0)   # strong coupling: synchronized

for c in (coupling_weak, coupling_mod, coupling_strong):
    brainstate.nn.init_all_states(c)
print('k values:', float(coupling_weak.k.value()),
      float(coupling_mod.k.value()), float(coupling_strong.k.value()))

k values: 0.009999999776482582 0.20000000298023224 1.0


**Finding optimal k:**

- Start with k ~ 0.1
- Increase until network shows desired synchronization
- Too high → all regions synchronized (unrealistic)
- Too low → no functional connectivity

## Multi-Modal Coupling

Couple different state variables:

In [5]:
# A model with several state variables can be coupled on each one separately.
# Here a Wilson-Cowan network couples its excitatory (rE) and inhibitory (rI)
# populations through different connectivity matrices.
M = 6
wc_nodes = brainmass.WilsonCowanStep(in_size=M)

rng = np.random.RandomState(0)
W_E = rng.rand(M, M) * 0.1; np.fill_diagonal(W_E, 0.0); W_E = jnp.asarray(W_E)
W_I = rng.rand(M, M) * 0.05; np.fill_diagonal(W_I, 0.0); W_I = jnp.asarray(W_I)

delays_wc = jnp.ones((M, M)) * (1.0 * u.ms)
src_idx_wc = np.tile(np.arange(M)[None, :], (M, 1))

# Separate diffusive coupling for the E and I populations
rE_src = wc_nodes.prefetch_delay('rE', delays_wc, src_idx_wc, init=braintools.init.Constant(0.0))
rE_self = wc_nodes.prefetch('rE')
rI_src = wc_nodes.prefetch_delay('rI', delays_wc, src_idx_wc, init=braintools.init.Constant(0.0))
rI_self = wc_nodes.prefetch('rI')
coupling_E = brainmass.DiffusiveCoupling(rE_src, rE_self, conn=W_E, k=0.2)
coupling_I = brainmass.DiffusiveCoupling(rI_src, rI_self, conn=W_I, k=0.1)

brainstate.nn.init_all_states(wc_nodes)
brainstate.nn.init_all_states(coupling_E)
brainstate.nn.init_all_states(coupling_I)

def multi_modal_step(i):
    with brainstate.environ.context(i=i, t=i * brainstate.environ.get_dt()):
        coupled_E = coupling_E.update()
        coupled_I = coupling_I.update()
        wc_nodes.update(rE_inp=coupled_E, rI_inp=coupled_I)
        return wc_nodes.rE.value

rE_activity = brainstate.transform.for_loop(multi_modal_step, np.arange(500))
rE_activity.shape  # (T, M)

(500, 6)

## Laplacian Coupling

Use graph Laplacian for diffusive coupling:

In [6]:
# Compute the (combinatorial) graph Laplacian L = W - D, where D = diag(row sums of W).
# Each row sums to zero, so L[i, i] = -sum_{j != i} W[i, j] and L[i, j] = W[i, j] for i != j.
L = brainmass.laplacian_connectivity(W, normalize=None)
print('row sums (~0):', float(jnp.abs(L.sum(axis=1)).max()))

# Diffusive coupling expressed via the Laplacian: C = -k * (L @ x).
k = 0.2
activity = jnp.asarray(np.random.RandomState(0).rand(N))
coupled = -k * (L @ activity)
coupled.shape  # (N,)

row sums (~0): 1.341104507446289e-07


(10,)

Normalized Laplacian:

In [7]:
# Symmetric normalization: D^(-1/2) W D^(-1/2) - I (pass normalize="sym").
# Use "rw" for the random-walk normalized form D^(-1) W - I.
L_norm = brainmass.laplacian_connectivity(W, normalize="sym")
L_norm.shape  # (N, N)

(10, 10)

## Time-Delayed Coupling

Implement axonal transmission delays:

In [8]:
# Axonal transmission delays are handled by ``prefetch_delay``: the coupling reads
# each source node's state as it was ``delays[i, j]`` ago, so no manual history
# buffer is needed. Here every connection has a 5 ms delay.
delay_nodes = brainmass.HopfStep(in_size=N, a=0.25, w=0.2)
delays_long = jnp.ones((N, N)) * (5.0 * u.ms)   # per-connection axonal delay
src_idx_d = np.tile(np.arange(N)[None, :], (N, 1))

x_delayed = delay_nodes.prefetch_delay('x', delays_long, src_idx_d,
                                       init=braintools.init.Constant(0.0))
x_local = delay_nodes.prefetch('x')
delayed_coupling = brainmass.DiffusiveCoupling(x_delayed, x_local, conn=W, k=0.2)

brainstate.nn.init_all_states(delay_nodes)
brainstate.nn.init_all_states(delayed_coupling)  # owns the delay buffer

def step_with_delay(i):
    with brainstate.environ.context(i=i, t=i * brainstate.environ.get_dt()):
        coupled = delayed_coupling.update()   # uses delayed source states
        delay_nodes.update(coupled, 0.0)
        return delay_nodes.x.value

delayed_activity = brainstate.transform.for_loop(step_with_delay, jnp.arange(1000))
delayed_activity.shape  # (T, N)

(1000, 10)

## Custom Coupling Functions

Implement custom coupling rules:

In [9]:
# A custom coupling rule is just a function of the source state and connectivity.
# This one only propagates activity from sources above a threshold.
def nonlinear_coupling(source, conn, k, threshold=0.5):
    """Only couple from source nodes whose activity exceeds ``threshold``."""
    active_source = jnp.where(source > threshold, source, 0.)
    return k * (conn @ active_source)

custom_nodes = brainmass.HopfStep(in_size=N, a=0.25, w=0.2)
brainstate.nn.init_all_states(custom_nodes)

def custom_network_step(i):
    with brainstate.environ.context(i=i, t=i * brainstate.environ.get_dt()):
        activity = custom_nodes.x.value
        coupled = nonlinear_coupling(activity, W, k=0.2, threshold=0.3)
        custom_nodes.update(coupled, 0.0)
        return custom_nodes.x.value

custom_activity = brainstate.transform.for_loop(custom_network_step, np.arange(500))
custom_activity.shape  # (T, N)

(500, 10)

## Directional Coupling

Different coupling for different directions:

In [10]:
# Asymmetric (directional) connectivity: feedforward and feedback have different
# weights and strengths. (Real connectomes would be loaded from disk; here we
# synthesize two random matrices with a fixed seed.)
rng = np.random.RandomState(0)
W_forward = rng.rand(N, N) * 0.1; np.fill_diagonal(W_forward, 0.0); W_forward = jnp.asarray(W_forward)
W_backward = rng.rand(N, N) * 0.05; np.fill_diagonal(W_backward, 0.0); W_backward = jnp.asarray(W_backward)

dir_nodes = brainmass.HopfStep(in_size=N, a=0.25, w=0.2)
src_idx_dir = np.tile(np.arange(N)[None, :], (N, 1))
delays_dir = jnp.ones((N, N)) * (1.0 * u.ms)

x_ff = dir_nodes.prefetch_delay('x', delays_dir, src_idx_dir, init=braintools.init.Constant(0.0))
x_fb = dir_nodes.prefetch_delay('x', delays_dir, src_idx_dir, init=braintools.init.Constant(0.0))
coupling_ff = brainmass.AdditiveCoupling(x_ff, conn=W_forward, k=0.3)
coupling_fb = brainmass.AdditiveCoupling(x_fb, conn=W_backward, k=0.1)

brainstate.nn.init_all_states(dir_nodes)
brainstate.nn.init_all_states(coupling_ff)
brainstate.nn.init_all_states(coupling_fb)

def directional_step(i):
    with brainstate.environ.context(i=i, t=i * brainstate.environ.get_dt()):
        ff_input = coupling_ff.update()      # feedforward coupling
        fb_input = coupling_fb.update()      # feedback coupling (weaker)
        total_coupling = ff_input + fb_input
        dir_nodes.update(total_coupling, 0.0)
        return dir_nodes.x.value

dir_activity = brainstate.transform.for_loop(directional_step, np.arange(500))
dir_activity.shape  # (T, N)

(500, 10)

## Optimization and Performance

### JIT Compilation

In [11]:
# Wrapping a step in ``brainstate.transform.jit`` compiles the coupling + node
# update together. (``brainstate.transform.for_loop`` already JIT-compiles its
# body, so this is mainly useful for a single fused step you call yourself.)
jit_nodes = brainmass.HopfStep(in_size=N, a=0.25, w=0.2)
x_src_jit = jit_nodes.prefetch_delay('x', delays, src_idx, init=braintools.init.Constant(0.0))
x_self_jit = jit_nodes.prefetch('x')
jit_coupling = brainmass.DiffusiveCoupling(x_src_jit, x_self_jit, conn=W, k=0.2)
brainstate.nn.init_all_states(jit_nodes)
brainstate.nn.init_all_states(jit_coupling)

@brainstate.transform.jit
def fast_network_step(i):
    with brainstate.environ.context(i=i, t=i * brainstate.environ.get_dt()):
        coupled = jit_coupling.update()
        jit_nodes.update(coupled, 0.0)
        return jit_nodes.x.value

# First call traces & compiles; subsequent calls reuse the compiled kernel.
_ = fast_network_step(0)
state_x = fast_network_step(1)
state_x.shape  # (N,)

(10,)

### Sparse Connectivity

For large sparse networks, consider sparse representations:

In [12]:
# For large networks most weights are tiny. Thresholding small weights to zero
# yields a sparser connectivity matrix (still stored densely here as (N, N)).
W_sparse = jnp.where(W > 0.01, W, 0.)
print('nonzero before:', int((W > 0).sum()), ' after threshold:', int((W_sparse > 0).sum()))

nonzero before: 90  after threshold: 90


## Best Practices

1. **Start Simple**: Test with diffusive coupling before trying custom coupling
2. **Normalize SC**: Divide by max or row/column sums to prevent instability
3. **Tune k Systematically**: Scan coupling strengths to find regime
4. **Monitor FC**: Compare simulated functional connectivity to empirical data
5. **Check Units**: Ensure coupling input has correct units for the model

## Troubleshooting

**Network Explodes:**

- Reduce coupling strength k
- Normalize connectivity matrix
- Add damping/noise

**No Synchronization:**

- Increase k
- Check connectivity topology (any isolated nodes?)
- Ensure sufficient simulation time

**Unphysical Dynamics:**

- Check coupling sign (diffusive should drive toward average)
- Verify connectivity matrix orientation (row vs column convention)

## Next Steps

- {doc}`/tutorials/05_forward_models` - Map coupled network to neuroimaging signals
- {doc}`/tutorials/06_fitting_with_gradients` - Optimize coupling parameters
- {doc}`../reference/coupling` - Full coupling API reference

## See Also

- {doc}`/tutorials/04_building_a_network` - Network construction
- {doc}`../gallery/index` - Coupling examples